# Aggregation Pipeline

## Overview
MongoDB's **aggregation pipeline** is the equivalent of SQL's GROUP BY, HAVING, window functions, and JOINs combined. A pipeline is a list of stages; each stage transforms the documents from the previous stage.

**Pipeline structure:**
```python
collection.aggregate([
    {"$stage_1": { ... }},
    {"$stage_2": { ... }},
    ...
])
```

Documents flow through the stages left-to-right (top-to-bottom). Each stage receives the output of the prior stage and passes its output to the next.

**Performance rule:** Put `$match` and `$limit` as early as possible to reduce the document count before expensive stages.

**SQL to aggregation pipeline:**

| SQL | Pipeline equivalent |
|---|---|
| `WHERE ...` | `{"$match": {...}}` |
| `GROUP BY col` | `{"$group": {"_id": "$col", ...}}` |
| `COUNT(*)` | `{"$sum": 1}` inside `$group` |
| `AVG(col)` | `{"$avg": "$col"}` inside `$group` |
| `HAVING count > 5` | `{"$match": {"count": {"$gt": 5}}}` after `$group` |
| `ORDER BY col DESC` | `{"$sort": {"col": -1}}` |
| `LIMIT N` | `{"$limit": N}` |
| `SELECT col AS alias` | `{"$project": {"alias": "$col"}}` |
| `LEFT JOIN` | `{"$lookup": {...}}` |
| `UNNEST(array)` | `{"$unwind": "$array_field"}` |

---

In [1]:

print("=== Aggregation pipeline overview ===")
stages = [
    ("$match",    "Filter documents (like WHERE or HAVING)",  "Early in pipeline to reduce documents"),
    ("$group",    "Group by key and aggregate",               "Like SQL GROUP BY + aggregate functions"),
    ("$project",  "Reshape documents: add/remove/rename/compute fields", "Like SQL SELECT column list"),
    ("$sort",     "Sort documents",                           "Like SQL ORDER BY"),
    ("$limit",    "Keep first N documents",                   "Like SQL LIMIT"),
    ("$skip",     "Skip N documents",                         "Like SQL OFFSET"),
    ("$unwind",   "Deconstruct array: one doc per element",   "Like SQL LATERAL JOIN / UNNEST"),
    ("$lookup",   "Left outer join from another collection",  "Like SQL LEFT JOIN"),
    ("$addFields","Add computed fields without removing others","Like SQL SELECT *, new_col AS ..."),
    ("$count",    "Count documents and output single doc",    "Like SQL SELECT COUNT(*)"),
    ("$facet",    "Run multiple sub-pipelines in parallel",   "Like SQL GROUPING SETS"),
    ("$bucket",   "Group into histogram buckets",             "Like SQL CASE WHEN ranges + GROUP BY"),
    ("$out",      "Write pipeline result to a collection",    "Like SQL CREATE TABLE AS SELECT"),
    ("$merge",    "Merge pipeline result into a collection",  "Like SQL INSERT ... ON CONFLICT"),
]
print(f"  {'Stage':<14s}  {'Purpose':<48s}  Tip")
print("  " + "-"*90)
for stage, purpose, tip in stages:
    print(f"  {stage:<14s}  {purpose:<48s}  {tip}")


=== Aggregation pipeline overview ===
  Stage           Purpose                                           Tip
  ------------------------------------------------------------------------------------------
  $match          Filter documents (like WHERE or HAVING)           Early in pipeline to reduce documents
  $group          Group by key and aggregate                        Like SQL GROUP BY + aggregate functions
  $project        Reshape documents: add/remove/rename/compute fields  Like SQL SELECT column list
  $sort           Sort documents                                    Like SQL ORDER BY
  $limit          Keep first N documents                            Like SQL LIMIT
  $skip           Skip N documents                                  Like SQL OFFSET
  $unwind         Deconstruct array: one doc per element            Like SQL LATERAL JOIN / UNNEST
  $lookup         Left outer join from another collection           Like SQL LEFT JOIN
  $addFields      Add computed fields without

---
## $match and $group: counting and aggregating

In [2]:

print("=== $match and $group: count encounters by province ===")
pipeline = '''
# How many patient encounters per province?
# SQL equivalent:
#   SELECT s.province, COUNT(e.encounter_id) AS encounter_count
#   FROM   encounters e JOIN patients p ON e.patient_id = p.patient_id
#   GROUP BY p.province
#   ORDER BY encounter_count DESC

pipeline = [
    # Stage 1: join patients into encounters (see $lookup notebook section)
    # For now assume encounters already have province embedded
    {"$group": {
        "_id":              "$province",        # GROUP BY province
        "encounter_count":  {"$sum": 1},        # COUNT(*)
        "unique_patients":  {"$addToSet": "$patient_id"},  # for COUNT DISTINCT later
    }},
    {"$addFields": {
        "unique_patient_count": {"$size": "$unique_patients"}
    }},
    {"$project": {
        "province":             "$_id",
        "encounter_count":      1,
        "unique_patient_count": 1,
        "_id":                  0
    }},
    {"$sort": {"encounter_count": -1}},
]

results = list(encounters.aggregate(pipeline))
for r in results:
    print(r)
'''
print(pipeline)

print("Example output:")
output = [
    {"province": "ON", "encounter_count": 12, "unique_patient_count": 4},
    {"province": "NB", "encounter_count": 9,  "unique_patient_count": 3},
    {"province": "BC", "encounter_count": 7,  "unique_patient_count": 3},
    {"province": "QC", "encounter_count": 5,  "unique_patient_count": 2},
]
import json
for r in output:
    print(json.dumps(r))


=== $match and $group: count encounters by province ===

# How many patient encounters per province?
# SQL equivalent:
#   SELECT s.province, COUNT(e.encounter_id) AS encounter_count
#   FROM   encounters e JOIN patients p ON e.patient_id = p.patient_id
#   GROUP BY p.province
#   ORDER BY encounter_count DESC

pipeline = [
    # Stage 1: join patients into encounters (see $lookup notebook section)
    # For now assume encounters already have province embedded
    {"$group": {
        "_id":              "$province",        # GROUP BY province
        "encounter_count":  {"$sum": 1},        # COUNT(*)
        "unique_patients":  {"$addToSet": "$patient_id"},  # for COUNT DISTINCT later
    }},
    {"$addFields": {
        "unique_patient_count": {"$size": "$unique_patients"}
    }},
    {"$project": {
        "province":             "$_id",
        "encounter_count":      1,
        "unique_patient_count": 1,
        "_id":                  0
    }},
    {"$sort": {"encounter_count": -

---
## $lookup: joining collections

In [3]:

print("=== $lookup: joining collections ===")
lookup_code = '''
# SQL equivalent:
#   SELECT e.encounter_id, e.encounter_date, e.encounter_type,
#          p.first_name, p.last_name, p.province
#   FROM   encounters e
#   LEFT JOIN patients p ON e.patient_id = p.patient_id
#   WHERE  e.encounter_type = 'inpatient'

pipeline = [
    {"$match": {"encounter_type": "inpatient"}},   # filter first (reduce docs)

    {"$lookup": {
        "from":         "patients",                # the collection to join
        "localField":   "patient_id",             # field in encounters
        "foreignField": "patient_id",             # field in patients
        "as":           "patient_info"            # name of the joined array
    }},
    # $lookup produces an array; use $unwind to flatten to one doc per match
    {"$unwind": "$patient_info"},

    {"$project": {
        "encounter_id":   1,
        "encounter_date": 1,
        "encounter_type": 1,
        "first_name":     "$patient_info.first_name",
        "last_name":      "$patient_info.last_name",
        "province":       "$patient_info.province",
        "_id":            0
    }},
    {"$sort": {"encounter_date": -1}},
]

results = list(encounters.aggregate(pipeline))
'''
print(lookup_code)

print("$lookup result structure BEFORE $unwind:")
before = {
    "encounter_id": "E001",
    "encounter_date": "2023-04-10",
    "encounter_type": "inpatient",
    "patient_id": "P001",
    "patient_info": [
        {"patient_id": "P001", "first_name": "Aroha", "last_name": "Ngata", "province": "NB"}
    ]
}
print(json.dumps(before, indent=2))

print()
print("$lookup result AFTER $unwind (patient_info is now an object, not an array):")
after = {
    "encounter_id": "E001",
    "encounter_date": "2023-04-10",
    "encounter_type": "inpatient",
    "patient_id": "P001",
    "patient_info": {"patient_id": "P001", "first_name": "Aroha", "last_name": "Ngata", "province": "NB"}
}
print(json.dumps(after, indent=2))


=== $lookup: joining collections ===

# SQL equivalent:
#   SELECT e.encounter_id, e.encounter_date, e.encounter_type,
#          p.first_name, p.last_name, p.province
#   FROM   encounters e
#   LEFT JOIN patients p ON e.patient_id = p.patient_id
#   WHERE  e.encounter_type = 'inpatient'

pipeline = [
    {"$match": {"encounter_type": "inpatient"}},   # filter first (reduce docs)

    {"$lookup": {
        "from":         "patients",                # the collection to join
        "localField":   "patient_id",             # field in encounters
        "foreignField": "patient_id",             # field in patients
        "as":           "patient_info"            # name of the joined array
    }},
    # $lookup produces an array; use $unwind to flatten to one doc per match
    {"$unwind": "$patient_info"},

    {"$project": {
        "encounter_id":   1,
        "encounter_date": 1,
        "encounter_type": 1,
        "first_name":     "$patient_info.first_name",
        "last_name":  

---
## $unwind and aggregation expressions

In [4]:

print("=== $unwind: expanding arrays ===")
unwind_code = '''
# A patient document has: "conditions": ["hypertension", "type2_diabetes"]
# $unwind produces one document per condition element.
# SQL equivalent: CROSS JOIN LATERAL UNNEST(conditions) AS cond

pipeline = [
    {"$unwind": "$conditions"},          # one doc per condition
    {"$group": {
        "_id":           "$conditions",  # GROUP BY condition
        "patient_count": {"$sum": 1}
    }},
    {"$sort": {"patient_count": -1}},
    {"$limit": 10},
]
results = list(patients.aggregate(pipeline))
'''
print(unwind_code)

print("Effect of $unwind on a document:")
original = {"patient_id": "P001", "first_name": "Aroha",
            "conditions": ["hypertension", "type2_diabetes"]}
print("Before $unwind:", json.dumps(original))
print("After  $unwind (2 documents):")
for cond in original["conditions"]:
    doc = {k: v for k, v in original.items() if k != "conditions"}
    doc["conditions"] = cond
    print("  ", json.dumps(doc))

print()
print("Aggregation expressions reference:")
exprs = [
    ("$sum",   "Sum of values; $sum: 1 counts documents"),
    ("$avg",   "Average"),
    ("$min",   "Minimum value"),
    ("$max",   "Maximum value"),
    ("$count", "Count documents (pipeline stage, not group accumulator)"),
    ("$push",  "Accumulate values into an array"),
    ("$addToSet", "Accumulate unique values into an array"),
    ("$first", "First value in group (requires $sort before $group)"),
    ("$last",  "Last value in group (requires $sort before $group)"),
    ("$concat","Concatenate strings: {'$concat': ['$first_name', ' ', '$last_name']}"),
    ("$cond",  "If/then/else: like SQL CASE WHEN"),
    ("$dateToString", "Format a date field"),
]
for expr, desc in exprs:
    print(f"  {expr:<16s}  {desc}")


=== $unwind: expanding arrays ===

# A patient document has: "conditions": ["hypertension", "type2_diabetes"]
# $unwind produces one document per condition element.
# SQL equivalent: CROSS JOIN LATERAL UNNEST(conditions) AS cond

pipeline = [
    {"$unwind": "$conditions"},          # one doc per condition
    {"$group": {
        "_id":           "$conditions",  # GROUP BY condition
        "patient_count": {"$sum": 1}
    }},
    {"$sort": {"patient_count": -1}},
    {"$limit": 10},
]
results = list(patients.aggregate(pipeline))

Effect of $unwind on a document:
Before $unwind: {"patient_id": "P001", "first_name": "Aroha", "conditions": ["hypertension", "type2_diabetes"]}
After  $unwind (2 documents):
   {"patient_id": "P001", "first_name": "Aroha", "conditions": "hypertension"}
   {"patient_id": "P001", "first_name": "Aroha", "conditions": "type2_diabetes"}

Aggregation expressions reference:
  $sum              Sum of values; $sum: 1 counts documents
  $avg              Average
  $

---
## Common Pitfalls

**1. Putting $match late in the pipeline**
Every stage processes all documents passed from the prior stage. A `$match` at stage 5 means stages 1-4 processed documents that will ultimately be discarded. Always put `$match` as early as possible -- ideally as the very first stage -- to reduce the working set before expensive operations like `$lookup` or `$group`.

**2. Forgetting $unwind after $lookup**
`$lookup` always produces an array field (even for a one-to-one join). If you forget `$unwind`, the joined data is nested in an array and `$project` fields like `'$patient_info.first_name'` return an array instead of a scalar value. Always follow `$lookup` with `$unwind` unless you intentionally want the array of joined documents.

**3. $group _id: null for global aggregates**
To aggregate across all documents (like SQL `SELECT COUNT(*) FROM table`), use `_id: null`: `{'$group': {'_id': null, 'total': {'$sum': 1}}}`. Using `_id: '$missing_field'` where the field does not exist groups all documents under `_id: null` too -- which can look correct but is actually a bug.

**4. Using $push instead of $addToSet for COUNT DISTINCT**
`$push` accumulates all values including duplicates. To count distinct values, use `$addToSet` and then `$size` on the resulting array: `{'$addToSet': '$patient_id'}` followed by `{'$addFields': {'unique_count': {'$size': '$patient_set'}}}`.

**5. $unwind on a missing or null array field drops documents by default**
If a document has no `conditions` field or `conditions: null`, `$unwind: '$conditions'` silently drops that document from the pipeline. Use `{'$unwind': {'path': '$conditions', 'preserveNullAndEmptyArrays': true}}` to keep documents with missing/null/empty arrays.

**6. Not using allowDiskUse for large aggregations**
By default, each aggregation pipeline stage is limited to 100 MB of RAM. Large `$group` or `$sort` operations fail with a memory exceeded error. Pass `allowDiskUse=True` to `aggregate()` to spill to disk when needed: `collection.aggregate(pipeline, allowDiskUse=True)`.


---
*sql_methods_library - Samantha McGarrigle*